## Recommendation System

In [123]:
# Importing required packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
pd.set_option('display.max_columns',200)

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

### 1. Data Preprocessing

In [5]:
# Loading Dataset
df=pd.read_csv('C:/Data Science/Assignments_Files/Recommendation System/anime.csv')
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [7]:
# Creating copy of dataset for backup
df_copy=df.copy()

In [9]:
# Explore the dataset to understand its structure and attributes.
# Checking size of dataset
df.shape

(12294, 7)

In [11]:
# Checking columns and there datatypes
df.dtypes

anime_id      int64
name         object
genre        object
type         object
episodes     object
rating      float64
members       int64
dtype: object

In [13]:
# Checking for Duplicates 
df.duplicated().sum()

0

In [15]:
# Checking for Null Values
df.isna().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [17]:
# Dropping null values for genre
df=df.dropna(subset=['genre'])

In [19]:
# Filling null values of type with mode
df.type.fillna(df.type.mode()[0],inplace=True)

In [27]:
# Handling null values of rating
df['rating'] = df['rating'].replace("Unknown", np.nan)
df['rating'] = df['rating'].astype(float)

df = df.dropna(subset=['rating'])

In [25]:
df.isna().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

### 2. Feature Extraction

In [33]:
# Converting categorical values into Numerical columns
df['genre'] = df['genre'].apply(lambda x: x.split(", "))

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre'])

genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)

In [37]:
# Take a look on Converted Data
genre_df.head()

,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,Harem,Hentai,Historical,Horror,Josei,Kids,Magic,Martial Arts,Mecha,Military,Music,Mystery,Parody,Police,Psychological,Romance,Samurai,School,Sci-Fi,Seinen,Shoujo,Shoujo Ai,Shounen,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Vampire,Yaoi,Yuri
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [39]:
# Selecting Numerical Features
numerical_features = df[['episodes', 'rating', 'members']]

In [41]:
# Normalizing Numerical data using MinMaxScaler
scaler = MinMaxScaler()
numerical_scaled = scaler.fit_transform(numerical_features)

numerical_df = pd.DataFrame(numerical_scaled, columns=numerical_features.columns)

ValueError: could not convert string to float: 'Unknown'

In [47]:
# While Standardizing We found that There is somewhere 'Unknown' value inside numerical column
# Checking 'Unknown' value in which column it is present
print(df['episodes'].unique())
print(df['rating'].unique())
print(df['members'].unique())

['1' '64' '51' '24' '10' '148' '110' '13' '201' '25' '22' '75' '4' '26'
 '12' '27' '43' '74' '37' '2' '11' '99' 'Unknown' '39' '101' '47' '50'
 '62' '33' '112' '23' '3' '94' '6' '8' '14' '7' '40' '15' '203' '77' '291'
 '120' '102' '96' '38' '79' '175' '103' '70' '153' '45' '5' '21' '63' '52'
 '28' '145' '36' '69' '60' '178' '114' '35' '61' '34' '109' '20' '9' '49'
 '366' '97' '48' '78' '358' '155' '104' '113' '54' '167' '161' '42' '142'
 '31' '373' '220' '46' '195' '17' '1787' '73' '147' '127' '16' '19' '98'
 '150' '76' '53' '124' '29' '115' '224' '44' '58' '93' '154' '92' '67'
 '172' '86' '30' '276' '59' '72' '330' '41' '105' '128' '137' '56' '55'
 '65' '243' '193' '18' '191' '180' '91' '192' '66' '182' '32' '164' '100'
 '296' '694' '95' '68' '117' '151' '130' '87' '170' '119' '84' '108' '156'
 '140' '331' '305' '300' '510' '200' '88' '1471' '526' '143' '726' '136'
 '1818' '237' '1428' '365' '163' '283' '71' '260' '199' '225' '312' '240'
 '1306' '1565' '773' '1274' '90' '475' '263' '8

In [49]:
# It is present inside Episodes columns
# Replacing Unknown values with null first
df['episodes'] = df['episodes'].replace("Unknown", np.nan)
#Converting it into float
df['episodes'] = df['episodes'].astype(float)
# Fill missing values
df['episodes'].fillna(df['episodes'].median(), inplace=True)

In [70]:
# Recreating Numerical Features again
numerical_features = df[['episodes', 'rating', 'members']]

In [72]:
# Again Normalizing Numerical Data
scaler = MinMaxScaler()
numerical_scaled = scaler.fit_transform(numerical_features)
numerical_df = pd.DataFrame(numerical_scaled, columns=numerical_features.columns)

In [74]:
# Combining all features
final_features = pd.concat([
    genre_df,
    numerical_df,
    df.filter(like='type_')
], axis=1)

In [76]:
# Final Features Matrix
print(final_features.shape)

(12284, 46)


### 3. Recommendation System

In [107]:
# Filling Null values of final_features with 0
final_features.fillna(0,inplace=True)

In [109]:
# Computing Similarity Matrix
similarity_matrix = cosine_similarity(final_features)

In [115]:
# Computing Indices
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [117]:
 def recommend_with_threshold(anime_name, df, similarity_matrix, indices, threshold=0.5):
    
    idx = indices[anime_name]
    
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Filter using threshold
    sim_scores = [i for i in sim_scores if i[1] >= threshold and i[0] != idx]
    
    # Sort descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df[['name', 'rating', 'genre']].iloc[anime_indices]

In [119]:
# Example Usage
recommend_with_threshold("Naruto", df, similarity_matrix, indices, threshold=0.6)

,name,rating,genre
615,Naruto: Shippuuden,7.94,"[Action, Comedy, Martial Arts, Shounen, Super ..."
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,7.68,"[Action, Comedy, Martial Arts, Shounen, Super ..."
486,Boruto: Naruto the Movie,8.03,"[Action, Comedy, Martial Arts, Shounen, Super ..."
1343,Naruto x UT,7.58,"[Action, Comedy, Martial Arts, Shounen, Super ..."
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,7.53,"[Action, Comedy, Martial Arts, Shounen, Super ..."
...,...,...,...
5816,Dr. Slump Movie 10: Arale no Bikkuriman,6.34,"[Comedy, Sci-Fi, Shounen]"
5846,Street Fighter: Aratanaru Kizuna,6.34,"[Action, Adventure, Shounen]"
6048,To Heart: Remember My Memories Specials,6.27,"[Comedy, Martial Arts, Parody, Samurai, Sci-Fi..."
5992,Baku Tech! Bakugan,6.28,"[Action, Game, Shounen]"


### 4. Evaluation

In [173]:
# Declaring train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_features = final_features.loc[train_df.index]

In [175]:
# Training Features
train_df = train_df.reset_index(drop=True)
train_features = train_features.reset_index(drop=True)

In [177]:
# Creating Similarity Matrix on train features
similarity_matrix = cosine_similarity(train_features)

In [179]:
indices = pd.Series(train_df.index, index=train_df['name']).drop_duplicates()

In [ ]:
# Defining Relevance
# Rating ≥ 7 → Relevant (Good anime)
# Rating < 7 → Not relevant

In [195]:
def evaluate_model(train_df, similarity_matrix, indices, k=10):
    
    precision_list = []
    recall_list = []
    
    # total relevant items in dataset
    total_relevant = len(train_df[train_df['rating'] >= 6])
    
    for anime in train_df['name']:
        
        if anime not in indices:
            continue
        
        idx = indices[anime]
        
        # similarity scores 
        sim_scores = list(enumerate(similarity_matrix[idx].ravel()))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:k+1]
        
        recommended_indices = [i[0] for i in sim_scores if i[0] < len(train_df)]
        
        if len(recommended_indices) == 0:
            continue
        
        recommended = train_df.iloc[recommended_indices]
        
        # relevant recommended
        relevant_recommended = recommended[recommended['rating'] >= 6]
        
        # Precision@K
        precision = len(relevant_recommended) / len(recommended)
        
        # Recall (approx)
        recall = len(relevant_recommended) / total_relevant if total_relevant > 0 else 0
        
        precision_list.append(precision)
        recall_list.append(recall)
    
    # averages
    avg_precision = sum(precision_list) / len(precision_list)
    avg_recall = sum(recall_list) / len(recall_list)
    
    # F1 Score
    if avg_precision + avg_recall == 0:
        f1 = 0
    else:
        f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)
    
    return avg_precision, avg_recall, f1

In [197]:
precision, recall, f1 = evaluate_model(train_df, similarity_matrix, indices)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.7517449089278903
Recall: 0.0010845038436939662
F1 Score: 0.002165883078898854


## Interview Questions

### 1. Can you explain the difference between user-based and item-based collaborative filtering?
 ### User-Based Collaborative Filtering
User-based collaborative filtering focuses on finding users with similar preferences and recommending items that those similar users liked.

**How it works:**
1. Identify users similar to the target user using similarity measures like cosine similarity.
2. Find items that similar users have liked.
3. Recommend those items to the target user.

**Example:**
If User A and User B have similar tastes, and User B liked an anime that User A has not seen, then that anime will be recommended to User# A.

---

### Item-Based Collaborative Filtering
Item-based collaborative filtering focuses on finding items similar to those the user has already liked and recommending those similar items.

**How it works:**
1. Identify items similar to the items the user has liked.
2. Recommend those similar items.

**Example:**
If a user liked "Naruto" and "Naruto" is similar to "Bleach", then "Bleach" will be recommended.

---

### Key Differences

| Feature        | User-Based Collaborative Filtering | Item-Based Collaborative Filtering |
|---------------|------------------------------------|------------------------------------|
| Focus         | Users                              | Items                              |
| Similarity    | Between users                      | Between items                      |
| Scalability   | Less scalable for large users      | More scalable                      |
| Use Case      | Smaller datasets                   | Large-sca#le systems                |

---

## 2. What is collaborati#ve filtering, and how does it work?

### Definition
Collaborative filtering is a recommendation technique that suggests items based on user behavior and preferences, without requiring inform#ation about the items themselves.

---

### Core Idea
Users who have similar preferences in the past are likely to h#ave similar preferences in the future.

---

### Types
1. User-based collaborative filterin#g  
2. Item-based collaborative filtering  

---

### How it works
1. Collect user-item interaction data such as ratings, clicks, or purchases.
2. Construct a user-item interaction matrix.
3. Compute similarity between users or items using methods like cosine similarity or Pearson correlation.
4. Predict missing values or preferences.
#5. Recommend items with the highest predicted scores.

---

### Example

| User | Naruto | One Piece | Bleach |
|------|--------|----------|--------|
| A    | 5      | 4        | ?      |
| B    | 5      | 4        | 5      |

Since User A and Use#r B have similar ratings, "Bleach" can be recommended to User A.

---

### Advantages
- Does not require item feature#s
- Provides personalized recommendations
- Learns from user behavior

---

### Limitations
- Cold start problem foehavior, either between users or between items, without relying on explicit content features.